# Step 4 — Competitor Comparison (before drafting)

Benchmark the company vs peers on the same metrics (`docs/competitor.md`). Peer values are
grounded `FinancialFact`s too, so the comparison is fully cited. **Lags feed the Predictive
Analyst**; **leads** are used proactively in the script.

In [1]:
import sys
from pathlib import Path
def _root():
    p = Path.cwd()
    for d in (p, *p.parents):
        if (d / "requirements.txt").exists():
            return d
    return p
ROOT = _root()
sys.path.insert(0, str(ROOT / "src"))
from ir_copilot.config import settings
print("backends -> qdrant:", settings.qdrant_mode, "| embeddings:", settings.embedding_backend,
      "| sentiment:", settings.sentiment_backend, "| llm:", settings.llm_backend)

backends -> qdrant: memory | embeddings: hash | sentiment: lexicon | llm: mock


In [2]:
from ir_copilot.facts import build_fact_store
from ir_copilot.agents.competitor import compare

store = build_fact_store(settings.ticker, settings.period, use_mock=settings.use_mock_data)
peers = {p: build_fact_store(p, settings.period, use_mock=settings.use_mock_data) for p in settings.peers}
pc = compare(store, peers)

print(f"{pc.company} vs peers {pc.peers}\n")
hdr = f"{'metric':<18}{pc.company:>10}" + "".join(f"{p:>10}" for p in pc.peers) + f"{'rank':>7}"
print(hdr); print("-" * len(hdr))
for m in pc.metrics:
    row = f"{m.metric:<18}{m.company_value:>10.2f}"
    row += "".join(f"{m.peer_values.get(p, float('nan')):>10.2f}" for p in pc.peers)
    row += f"{m.rank:>7}"
    print(row)

print(f"\nLEADS: {pc.leads}")
print(f"LAGS : {pc.lags}  (-> become predicted questions)")
print("provenance e.g.:", pc.metrics[0].metric, "fact_ids", pc.metrics[0].fact_ids)
assert pc.metrics, "expected peer metrics"

NVDA vs peers ['AMD', 'TSLA']

metric                  NVDA       AMD      TSLA   rank
-------------------------------------------------------
gross_margin           74.93     52.82     21.08      1
operating_margin       65.60     14.40      4.20      1
roe                    33.06      2.17      0.57      1
roic                   31.43      2.08      0.60      1
roa                    25.02      1.77      0.34      1
ttm_eps                 6.53      3.05      1.09      1

LEADS: ['gross_margin', 'operating_margin', 'roe', 'roic', 'roa', 'ttm_eps']
LAGS : []  (-> become predicted questions)
provenance e.g.: gross_margin fact_ids ['F-0013', 'F-0013', 'F-0013']


**Next (Step 5):** predict the hardest investor questions.